# Análise exploratória: internações SUS no estado de São Paulo (2024–2025)

Este notebook documenta a exploração que deu origem ao **catálogo de perguntas** do chatbot.
Fontes:
- **SIH/SUS (RD-SP)**: uma linha por AIH; base relacional (`fato_internacao`, `agg_mensal_*`).
- **CNES via API de Dados Abertos (JSON)**: hospitais, UBS e unidades de urgência, com atributos variáveis em `atributos_json`.
- **CNES-LT**: leitos existentes e leitos SUS por estabelecimento e competência.
- **IBGE**: população estimada; **DATASUS**: divisão territorial (DRS, Regiões de Saúde, RRAS).

Pré-requisito: `python -m etl.build_db` e `python -m analysis.run_all` já executados.

In [1]:
import sqlite3, pandas as pd, sys, os
sys.path.insert(0, os.path.abspath('..'))
from etl.config import DB_PATH
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
con = sqlite3.connect(DB_PATH)
q = lambda s, p=(): pd.read_sql(s, con, params=p)
q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY 1")

,name
0,agg_mensal_hosp
1,agg_mensal_res
2,cnes_raw
3,dim_cid_capitulo
4,dim_estabelecimento
5,dim_municipio
6,fato_internacao
7,fato_leitos
8,res_capacidade_drs_mensal
9,res_cluster_meta


## 1. Volume e qualidade dos dados

In [2]:
q('''SELECT COUNT(*) AS aihs, SUM(conta_internacao) AS internacoes,
          SUM(ident = 5) AS aih_longa_permanencia,
          SUM(cap_cid IS NULL) AS sem_cid, SUM(idade IS NULL) AS sem_idade,
          SUM(cnes NOT IN (SELECT cnes FROM dim_estabelecimento)) AS sem_cadastro_api
       FROM fato_internacao''')

,aihs,internacoes,aih_longa_permanencia,sem_cid,sem_idade,sem_cadastro_api
0,5754750,5693745,61005,0,1,27635


A API do CNES cobre quase todos os hospitais do SIH. Os poucos que faltam (desativados ou de outra tipologia) aparecem como `CNES <código>`.

**Dados semiestruturados**: os campos que variam por estabelecimento ficam em JSON e são consultados com `json_extract`:

In [3]:
q('''SELECT grupo, json_extract(atributos_json, '$.descricao_turno_atendimento') AS turno, COUNT(*) n
       FROM dim_estabelecimento GROUP BY 1, 2 ORDER BY 1, 3 DESC''').groupby('grupo').head(3)

,grupo,turno,n
0,Hospital,ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:...,1257
1,Hospital,"ATENDIMENTO NOS TURNOS DA MANHA, TARDE E NOITE",175
2,Hospital,ATENDIMENTOS NOS TURNOS DA MANHA E A TARDE,172
6,UBS,ATENDIMENTOS NOS TURNOS DA MANHA E A TARDE,4980
7,UBS,"ATENDIMENTO NOS TURNOS DA MANHA, TARDE E NOITE",994
8,UBS,ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:...,168
13,Urgência,ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:...,610
14,Urgência,"ATENDIMENTO NOS TURNOS DA MANHA, TARDE E NOITE",70
15,Urgência,ATENDIMENTOS NOS TURNOS DA MANHA E A TARDE,29


## 2. Sazonalidade: por período e por região

In [4]:
m = q("SELECT mes_inter, SUM(internacoes) n FROM agg_mensal_hosp GROUP BY 1")
m.set_index('mes_inter')['n'].T.to_frame().T

mes_inter,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,...,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
n,220883,214143,236429,245113,244748,232809,235652,238726,238029,245898,...,244791,245441,254430,241253,247941,247892,247658,252105,232118,225011


In [5]:
s = q("SELECT chave, mes, indice FROM res_sazonalidade WHERE nivel = 'Capítulo CID'")
cap = q("SELECT cap_cod chave, cap_nome FROM dim_cid_capitulo")
p = s.merge(cap, on='chave').pivot_table(index='cap_nome', columns='mes', values='indice')
p['amplitude'] = p.max(axis=1) - p.min(axis=1)
p.sort_values('amplitude', ascending=False).round(2).head(8)

mes,1,2,3,4,5,6,7,8,9,10,11,12,amplitude
cap_nome,,,,,,,,,,,,,
Códigos especiais,NaN,0.94,0.74,0.92,1.72,1.27,1.05,0.92,1.13,0.50,0.65,1.09,1.22
Aparelho respiratório,0.69,0.74,0.95,1.17,1.29,1.23,1.06,1.08,1.02,1.00,0.94,0.82,0.60
Olho e anexos,0.84,0.88,0.92,0.95,1.03,1.01,1.08,1.12,1.11,1.28,1.05,0.73,0.55
Infecciosas e parasitárias,0.98,1.08,1.35,1.29,1.27,0.93,0.85,0.85,0.86,0.86,0.82,0.87,0.53
Ouvido,0.91,0.77,0.97,1.01,0.99,1.05,0.99,0.99,1.16,1.20,1.05,0.91,0.43
Malformações congênitas,0.98,0.99,0.96,1.02,1.01,0.99,1.04,1.08,1.04,1.10,0.97,0.82,0.28
Osteomuscular,0.88,0.88,0.94,1.00,1.03,0.99,1.07,1.07,1.13,1.14,1.02,0.86,0.28
Contatos com serviços de saúde,0.97,0.96,1.03,1.03,1.07,1.00,1.04,1.01,1.03,1.06,0.96,0.84,0.23


**Achados**
- Doenças **respiratórias** têm a maior sazonalidade: pico em maio (cerca de +29%) e vale em janeiro (cerca de −31%).
- **Infecciosas** concentram-se em março–maio, compatível com as epidemias de dengue de 2024–2025.
- Cirurgias eletivas (olho, osteomuscular) caem em dezembro–fevereiro e têm pico em outubro.
- O total de internações varia pouco (cerca de ±7%). A sazonalidade aparece por causa, não no volume agregado.

→ Perguntas: *"Como as internações variam ao longo dos meses?"*, *"Quais causas têm maior sazonalidade?"*, *"Como a sazonalidade das doenças respiratórias varia entre as DRS?"*

In [6]:
d = q("SELECT chave drs, mes, indice FROM res_sazonalidade WHERE nivel='DRS × Capítulo' AND cap_cid='X'")
d.pivot_table(index='drs', columns='mes', values='indice').round(2)

mes,1,2,3,4,5,6,7,8,9,10,11,12
drs,,,,,,,,,,,,
DRS I - Grande São Paulo,0.67,0.75,1.01,1.22,1.23,1.15,1.00,1.08,1.01,1.04,1.01,0.82
DRS II - Araçatuba,0.67,0.69,0.86,1.21,1.40,1.26,1.09,1.06,1.01,0.94,0.91,0.90
DRS III - Araraquara,0.67,0.67,0.92,1.12,1.35,1.37,1.12,1.06,1.09,0.96,0.86,0.79
DRS IV - Baixada Santista,0.81,0.74,1.03,1.18,1.24,1.19,0.97,1.07,1.01,1.03,0.94,0.79
DRS IX - Marília,0.68,0.67,0.89,1.04,1.33,1.34,1.13,1.09,1.08,0.96,0.92,0.86
DRS V - Barretos,0.74,0.70,0.88,1.25,1.33,1.29,1.07,1.06,1.08,0.91,0.84,0.84
DRS VI - Bauru,0.63,0.68,0.90,1.16,1.31,1.31,1.10,1.10,1.03,1.01,0.92,0.86
DRS VII - Campinas,0.70,0.79,0.96,1.18,1.29,1.25,1.11,1.10,0.98,0.94,0.91,0.79
DRS VIII - Franca,0.72,0.80,0.92,1.13,1.44,1.32,1.01,0.96,1.01,1.10,0.84,0.75


## 3. Rankings e comparações

In [7]:
q('''SELECT nome, drs_nome, ROUND(pop) pop, ROUND(taxa_int_1k,1) taxa_1k, ROUND(perm_media,1) perm,
          ROUND(pct_fora,1) pct_fora, ROUND(leitos_sus_10k,1) leitos_10k
       FROM res_municipio WHERE pop >= 20000 ORDER BY taxa_int_1k DESC LIMIT 10''')

,nome,drs_nome,pop,taxa_1k,perm,pct_fora,leitos_10k
0,Cajuru,DRS XIII - Ribeirão Preto,24225.0,171.4,3.8,28.1,22.7
1,Monte Aprazível,DRS XV - São José do Rio Preto,22659.0,149.5,3.5,45.4,13.2
2,Martinópolis,DRS XI - Presidente Prudente,25270.0,134.6,3.7,46.3,15.1
3,Paraguaçu Paulista,DRS IX - Marília,42068.0,127.2,3.3,15.7,21.2
4,Tanabi,DRS XV - São José do Rio Preto,26046.0,124.5,4.3,62.9,11.5
5,Ilhabela,DRS XVII - Taubaté,36444.0,122.9,2.9,23.9,13.6
6,Rancharia,DRS XI - Presidente Prudente,29346.0,119.4,3.5,26.6,27.6
7,Bastos,DRS IX - Marília,21918.0,116.4,3.3,50.2,8.7
8,Itararé,DRS XVI - Sorocaba,45270.0,113.3,2.6,22.6,11.7
9,Santa Rosa de Viterbo,DRS XIII - Ribeirão Preto,23718.0,111.8,3.8,60.4,8.0


In [8]:
q('''SELECT nome, municipio, ROUND(internacoes_ano) int_ano, ROUND(perm_media,1) perm,
          ROUND(leitos_sus) leitos_sus, ROUND(ocupacao_est) ocup_pct, ROUND(pct_atraidos) pct_de_fora
       FROM res_hospital ORDER BY internacoes DESC LIMIT 10''')

,nome,municipio,int_ano,perm,leitos_sus,ocup_pct,pct_de_fora
0,Hc Da Fmusp Hospital Das Clinicas Sao Paulo,São Paulo,57646.0,5.8,1476.0,62.0,43.0
1,Hospital De Base De Sao Jose Do Rio Preto,São José do Rio Preto,54245.0,5.0,811.0,92.0,81.0
2,Hospital Das Clinicas Faepa Ribeirao Preto,Ribeirão Preto,49998.0,4.6,851.0,74.0,63.0
3,Hosp Sta Marcelina Sao Paulo,São Paulo,32526.0,5.5,549.0,89.0,25.0
4,Hospital Das Clinicas Da Unicamp De Campinas,Campinas,31129.0,5.9,629.0,80.0,70.0
5,Hospital Municipal Dr Jose De Carvalho Florence,São José dos Campos,29228.0,4.0,389.0,83.0,9.0
6,Hospital Domingos Leonardo Ceravolo Presidente...,Presidente Prudente,28655.0,5.3,444.0,93.0,63.0
7,Santa Casa De Sao Paulo Hospital Central Sao P...,São Paulo,27833.0,6.0,1079.0,43.0,25.0
8,Hospital Das Clinicas Da Faculdade De Medicina...,Botucatu,27574.0,5.8,518.0,85.0,57.0
9,Hospital Sao Paulo Hospital De Ensino Da Unifesp,São Paulo,25405.0,4.2,797.0,37.0,42.0


In [9]:
q('''SELECT nome, ROUND(atraidos/2) atraidos_ano, ROUND(pct_atraidos,1) pct, ROUND(saldo_fluxo) saldo
       FROM res_municipio ORDER BY atraidos DESC LIMIT 10''')

,nome,atraidos_ano,pct,saldo
0,São Paulo,129548.0,17.6,107346.0
1,São José do Rio Preto,49572.0,56.6,49143.0
2,Ribeirão Preto,43412.0,48.8,42001.0
3,Campinas,30972.0,35.1,28685.0
4,Sorocaba,27311.0,45.3,25937.0
5,Presidente Prudente,25865.0,62.4,25363.0
6,Barretos,21239.0,67.5,20275.0
7,Bauru,18630.0,44.6,17180.0
8,Jundiaí,18422.0,40.6,17246.0
9,São José dos Campos,17777.0,31.3,16567.0


**Achados**
- Cerca de 30% das internações acontecem fora do município de residência. Capital, São José do Rio Preto, Ribeirão Preto e Campinas são os grandes polos.
- Taxas por habitante muito altas aparecem em municípios-sede de hospitais regionais (ex.: Barretos, Botucatu, Jaú).

→ Perguntas de ranking (municípios, hospitais, permanência), comparação entre municípios, panorama de um município e polos de atração.

## 4. Indicadores de capacidade

In [10]:
q('''SELECT drs_nome, ROUND(SUM(leitos_sus)*10000/SUM(pop),1) leitos_10k,
          ROUND(SUM(ubs)*10000/SUM(pop),2) ubs_10k,
          ROUND(SUM(dias_occ)/2*100/(SUM(leitos_sus)*365),1) ocupacao_est,
          ROUND(SUM(int_occ_ano)*100/SUM(int_res_ano),1) cobertura
       FROM res_municipio GROUP BY 1 ORDER BY leitos_10k DESC''')

,drs_nome,leitos_10k,ubs_10k,ocupacao_est,cobertura
0,DRS XIV - São João da Boa Vista,27.1,2.38,30.5,91.4
1,DRS V - Barretos,22.6,2.57,58.1,132.9
2,DRS IX - Marília,21.5,3.17,45.4,96.4
3,DRS XI - Presidente Prudente,20.7,3.79,56.3,98.6
4,DRS VI - Bauru,20.5,2.47,54.5,108.3
5,DRS XV - São José do Rio Preto,17.6,2.44,62.0,103.7
6,DRS XIII - Ribeirão Preto,17.4,1.73,52.3,108.4
7,DRS II - Araçatuba,14.7,2.25,47.0,86.9
8,DRS XII - Registro,14.3,4.79,51.8,96.1
9,DRS VIII - Franca,14.2,2.29,49.4,91.8


A ocupação estimada (dias de permanência ÷ leitos SUS × 365) é uma aproximação. Valores acima de 100% em hospitais indicam uso de leitos não-SUS ou cadastro desatualizado.

→ Perguntas: internações por leito, ocupação dos hospitais, estrutura hospitalar por DRS.

## 5. Padrões, agrupamentos e fatores de demanda

In [11]:
q("SELECT * FROM res_cluster_meta")

,k,silhueta,pc1_var,pc2_var,features
0,3,0.184975,0.328913,0.181687,"Internações por 1.000 hab., Permanência média ..."


In [12]:
q('''SELECT cluster, nome, n_municipios, ROUND(populacao) populacao, ROUND(taxa_int_1k,1) taxa,
          ROUND(perm_media,2) perm, ROUND(pct_fora,1) pct_fora, ROUND(leitos_sus_10k,1) leitos_10k,
          ROUND(ubs_10k,2) ubs_10k FROM res_cluster_perfil''')

,cluster,nome,n_municipios,populacao,taxa,perm,pct_fora,leitos_10k,ubs_10k
0,1,↑ % urgência · ↓ permanência · ↓ custo médio,159,3818546.0,102.1,3.80,49.0,22.9,3.71
1,2,↑ internação fora · ↑ sazonalidade · ↓ leitos/...,318,2364137.0,83.6,4.19,98.8,1.5,5.14
2,3,↓ % idosos · ↑ porte · ↓ UBS/hab.,168,39844816.0,63.5,4.46,43.3,12.5,1.73


In [13]:
q('''SELECT rotulo, ROUND(spearman,2) spearman, ROUND(coef,2) coef, ROUND(ci_low,2) ic_inf,
          ROUND(ci_high,2) ic_sup, ROUND(p_coef,4) p, ROUND(r2,2) r2, n FROM res_fatores''')

,rotulo,spearman,coef,ic_inf,ic_sup,p,r2,n
0,Porte do município (log da população),-0.37,-0.38,-1.02,0.27,0.2538,0.48,496
1,% internados fora do município,-0.05,-0.28,-0.61,0.05,0.0964,0.48,496
2,UBS por 10 mil hab.,0.41,0.20,0.02,0.39,0.0331,0.48,496
3,Leitos SUS por 10 mil hab.,0.32,0.15,0.06,0.23,0.0010,0.48,496
4,% causas respiratórias,0.38,0.13,-0.02,0.29,0.0969,0.48,496
5,% de internações de idosos (60+),0.45,0.11,0.03,0.19,0.0075,0.48,496
6,Amplitude sazonal,0.15,-0.09,-0.20,0.02,0.1120,0.48,496
7,% urgência,0.21,0.05,-0.04,0.15,0.2611,0.48,496
8,Densidade demográfica (log),-0.45,-0.01,-0.25,0.23,0.9201,0.48,496


**Achados**
- Três perfis de municípios aparecem com clareza: (1) sedes com leitos e alta taxa de internação, (2) municípios pequenos que internam quase tudo fora, (3) grandes centros urbanos com taxa menor e população mais jovem.
- Na regressão (R² ≈ 0,48), **leitos SUS/hab.**, **UBS/hab.** e **% de idosos** estão associados positivamente à taxa de internação. Uma hipótese é que a oferta revele demanda reprimida onde há pouca estrutura. A relação é associativa, não causal.

→ Perguntas: *"Quais grupos de municípios têm perfis parecidos?"*, *"Qual a relação entre UBS e internações?"*, *"Quais fatores estão associados ao aumento da demanda?"*